# 第14回 説明可能性と社会 — 「なぜそう判断したのか」を、誰に、どう示すのか

**このノートブックは第14回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 紙1枚に描ける決定木と、葉が数十万枚の森を比べる
2. **同じモデルに3つの説明方法を使い、答えが食い違うことを確かめる**
3. 性別を入力から外しても、判定が分かれることを確かめる
4. **合格率を揃えてみて、偏りが「消える」のか「移る」のかを見る**

所要時間:実習 約25分 / 課題 約40分

> 使うのは米国国勢調査データ(1994年)です。いまの社会をそのまま表すものではありません。

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。

> 使うのは米国国勢調査データ(1994年)です。**いまの社会をそのまま表すものではありません。**
> 考えるための材料として使います。

In [ ]:
# 準備(このセルを最初に1回だけ実行。初回は1〜2分かかります)
!pip install -q japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, japanize_matplotlib
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
plt.rcParams["font.size"] = 12
print("準備できました")

In [ ]:
# 米国国勢調査データ(1994年)を読み込む
# 年齢・学歴・職種などから「年収が5万ドルを超えるか」を予測します。
元 = fetch_openml("adult", version=2, as_frame=True)
df = 元.frame.dropna()
y  = (df["class"] == ">50K").astype(int)
性別 = (df["sex"] == "Male").astype(int)          # ← 予測には使いません(あとで検査に使います)

# 入力に使う9項目。性別も人種も入れません。
使う項目 = ["age", "workclass", "education-num", "marital-status", "occupation",
            "relationship", "hours-per-week", "capital-gain", "capital-loss"]
X = df[使う項目].copy()
文字列 = X.select_dtypes(include=["category", "object"]).columns
X[文字列] = OrdinalEncoder().fit_transform(X[文字列].astype(str))

Xtr, Xte, ytr, yte, 性tr, 性te = train_test_split(
    X, y, 性別, test_size=0.3, random_state=0, stratify=y)

print(f"全体 {len(df):,} 件   訓練 {len(Xtr):,} / テスト {len(Xte):,}")
print(f"高収入(>50K)の割合: {y.mean()*100:.1f}%")
print(f"男女の内訳: 男 {(性別==1).sum():,} / 女 {(性別==0).sum():,}")
print()
print("※ 1994年のデータです。いまの社会をそのまま表すものではありません。")
print("   考えるための材料として使います。")

## 1. 読める木と、読めない森

「AI は中身がブラックボックスだが、そのぶん精度が高い」── よく言われます。
**本当でしょうか。**

紙1枚に描ける決定木と、葉が数十万枚あるランダムフォレストを比べます。

In [ ]:
# ===== ① 読める木と、読めない森 =====
候補 = [("決定木 深さ3", DecisionTreeClassifier(max_depth=3, random_state=0)),
        ("決定木 深さ5", DecisionTreeClassifier(max_depth=5, random_state=0)),
        ("決定木 深さ8", DecisionTreeClassifier(max_depth=8, random_state=0)),
        ("ランダムフォレスト 100本", RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1))]

print(f"{'モデル':<26}{'葉の数':>12}{'正解率':>10}")
print("-" * 50)
学習済み = {}
for 名, m in 候補:
    m.fit(Xtr, ytr)
    学習済み[名] = m
    葉 = m.get_n_leaves() if hasattr(m, "get_n_leaves") else sum(t.get_n_leaves() for t in m.estimators_)
    print(f"{名:<26}{葉:>12,}{m.score(Xte, yte)*100:>9.2f}%")

木3 = 学習済み["決定木 深さ3"]; 森 = 学習済み["ランダムフォレスト 100本"]
print(f"\n深さ3の木と森の差: {森.score(Xte,yte)*100 - 木3.score(Xte,yte)*100:.2f} ポイント")
print("この差のために、説明を捨てる価値はあるでしょうか。")

In [ ]:
# 葉8枚の木は、実際にこう読めます
print(export_text(木3, feature_names=使う項目, max_depth=3))
print("読めますよね。そして、この判断が妥当かどうかを議論できます。")
print(f"森は葉が {sum(t.get_n_leaves() for t in 森.estimators_):,} 枚。誰にも読めません。")

> **この差を、どう考えるか**
>
> 「わずかな精度のために、説明を捨てるか」── これは技術ではなく、**判断**の問題です。
> 融資の審査なら説明義務があるかもしれません。広告の出し分けなら精度を取ってもよいでしょう。
>
> ただし、画像や言語では話が変わります(第6回・第8回)。
> 「常に読めるモデルで足りる」ではなく、**「確かめてから決めよ」**ということです。

## 2. 説明の方法を変えると、答えは変わるか

**まったく同じモデル・同じ予測**に対して、標準的な3つの方法で「どの項目が効いているか」を出します。

- **① 不純度** … 木の内部で、その項目が何回・どれだけ効果的に分岐に使われたか
- **② 並べ替え** … その列をぐちゃぐちゃにすると、正解率がどれだけ落ちるか
- **③ 線形係数** … まったく別のモデルを当てはめたときの係数の大きさ

どれも実務で普通に使われる方法です。答えは一致するでしょうか。

In [ ]:
# ===== ② 説明の方法を変えると、答えは変わるか =====
# まったく同じモデル(森)に対して、3つの標準的な方法で「重要度」を出します。

# ① 不純度による重要度(木の内部指標)
重要1 = pd.Series(森.feature_importances_, index=使う項目)

# ② 並べ替え重要度(その列を壊すと正解率がどれだけ落ちるか)
pi = permutation_importance(森, Xte, yte, n_repeats=5, random_state=0, n_jobs=-1)
重要2 = pd.Series(pi.importances_mean, index=使う項目)

# ③ 線形モデルの係数(まったく別のモデルで説明する)
sc = StandardScaler().fit(Xtr)
lr = LogisticRegression(max_iter=2000).fit(sc.transform(Xtr), ytr)
重要3 = pd.Series(np.abs(lr.coef_[0]), index=使う項目)

順位 = pd.DataFrame({
    "① 不純度":   重要1.rank(ascending=False).astype(int),
    "② 並べ替え": 重要2.rank(ascending=False).astype(int),
    "③ 線形係数": 重要3.rank(ascending=False).astype(int),
})
順位["食い違い"] = 順位.max(axis=1) - 順位.min(axis=1)
print(順位.sort_values("① 不純度").to_string())
print()
print(f"1位になった項目: ① {重要1.idxmax()} / ② {重要2.idxmax()} / ③ {重要3.idxmax()}")
print(f"食い違いが最大: {順位['食い違い'].idxmax()}(差 {順位['食い違い'].max()} 位)")

In [ ]:
# 図で見る
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, (名, s, c) in zip(ax, [("① 不純度", 重要1, "#3A46A8"),
                              ("② 並べ替え", 重要2, "#5B6BD6"),
                              ("③ 線形係数", 重要3, "#E08A2B")]):
    s2 = s.sort_values()
    a.barh(range(len(s2)), s2.values, color=c)
    a.set_yticks(range(len(s2))); a.set_yticklabels(s2.index, fontsize=9)
    a.set_title(名); a.grid(alpha=.3, axis="x")
plt.suptitle("同じモデル・同じ予測。違うのは説明の方法だけ", fontsize=12)
plt.tight_layout(); plt.show()

print("3つの図は、同じモデルを説明したものです。並び順が違いますね。")
print()
print("なぜ食い違うのか ── どれも「別のこと」を測っているからです。")
print("  ① 木がその項目で何回分岐したか")
print("  ② その列を壊すと正解率がどれだけ落ちるか")
print("  ③ まったく別のモデルでの係数の大きさ")
print()
print("どれかが間違っているのではありません。問いが違うのです。")
print("→「重要度を出しました」と言われたら、「どの方法ですか」と聞いてください。")

## 3. 性別を使わなければ、公平になるのか

入力の9項目に、**性別も人種も入れていません**。
「使っていないのだから差別ではない」── そう言えるでしょうか。

In [ ]:
# ===== ③ 性別を使わなければ、公平になるのか =====
# 入力の9項目に、性別も人種も入れていません。それでも結果は公平でしょうか。
予測 = 森.predict(Xte)

print(f"{'':<8}{'高収入と判定':>14}{'実際に高収入':>14}{'人数':>10}")
print("-" * 48)
for 名, 印 in [("男性", 1), ("女性", 0)]:
    g = 性te == 印
    print(f"{名:<8}{予測[g].mean()*100:>13.1f}%{yte[g].mean()*100:>13.1f}%{g.sum():>10,}")

判定比 = 予測[性te==0].mean() / 予測[性te==1].mean() * 100
実際比 = yte[性te==0].mean() / yte[性te==1].mean() * 100
print(f"\n女 ÷ 男:  モデルの判定 {判定比:.1f}%   データの実際 {実際比:.1f}%")
print("→ モデルは差を作り出したのではなく、データにある差をほぼそのまま再現しています。")

# 性別が本当に「消えている」か検査する
性別当て = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(Xtr, 性tr)
基準 = max(性te.mean(), 1 - 性te.mean()) * 100
print(f"\n【検査】同じ9項目から「性別」を予測させると… 正解率 {性別当て.score(Xte, 性te)*100:.1f}%")
print(f"        (でたらめに多数派を答えると {基準:.1f}%)")
print()
print("続柄・職種・週労働時間などに、性別の情報が色濃く残っています。")
print("→「入力から外す」ことは、「影響を消す」ことではありません。")

> **モデルは差別を「作り出した」のでしょうか**
>
> 上の出力で、**モデルの判定の比**と**データの実際の比**を比べてみてください。
> ほぼ同じなら、モデルは差を作り出したのではなく、**データにある差を再現している**ということです。
>
> これは「モデルが悪い」で片づく話ではありません。
> では、どうすればよいのでしょうか。「揃えればよい」と思いますよね。次で実際に揃えてみます。

## 4. では、揃えればよいのか

男女で「高収入と判定される割合」が同じになるように、**しきい値を男女別に調整**します。
男性には高めの基準、女性には低めの基準を使う、ということです。

In [ ]:
# ===== ④ では、揃えればよいのか =====
確率 = 森.predict_proba(Xte)[:, 1]
そのまま = (確率 >= 0.5).astype(int)

# 男女別のしきい値で「合格率」を揃える
目標 = そのまま.mean()
し男 = np.quantile(確率[性te==1], 1-目標)
し女 = np.quantile(確率[性te==0], 1-目標)
揃えた = np.where(性te==1, 確率 >= し男, 確率 >= し女).astype(int)

def 表示(名, p):
    return (f"{名:<24}{(p==yte).mean()*100:>8.2f}%"
            f"{p[性te==1].mean()*100:>11.1f}%{p[性te==0].mean()*100:>11.1f}%"
            f"{p[性te==0].mean()/p[性te==1].mean()*100:>9.1f}%")

print(f"{'方針':<24}{'正解率':>9}{'男の合格率':>12}{'女の合格率':>12}{'女÷男':>10}")
print("-" * 68)
print(表示("そのまま(0.5)", そのまま))
print(表示("合格率を揃える", 揃えた))
print(f"\n正解率は {(そのまま==yte).mean()*100:.2f}% → {(揃えた==yte).mean()*100:.2f}% "
      f"({(そのまま==yte).mean()*100-(揃えた==yte).mean()*100:.2f}ポイント減)")
print("うまくいったように見えます。── でも、誤りの中身を見てください。")

In [ ]:
# 誤りの内訳を、揃える前と後で比べる
def 誤り(p, 印):
    g = 性te == 印
    誤判定 = ((p==1) & (yte==0) & g).sum() / max(((yte==0) & g).sum(), 1) * 100
    見逃し = ((p==0) & (yte==1) & g).sum() / max(((yte==1) & g).sum(), 1) * 100
    return 誤判定, 見逃し

print(f"{'':<14}{'誤判定(男)':>12}{'誤判定(女)':>12}{'見逃し(男)':>12}{'見逃し(女)':>12}")
print("-" * 62)
記録 = {}
for 名, p in [("そのまま", そのまま), ("揃えたあと", 揃えた)]:
    a, b = 誤り(p, 1); c, d = 誤り(p, 0)
    記録[名] = (a, c, b, d)
    print(f"{名:<14}{a:>11.1f}%{c:>11.1f}%{b:>11.1f}%{d:>11.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
x = np.arange(2); w = 0.35
for a, (i, j, 題) in zip(ax, [(0, 1, "低収入なのに合格と誤判定"), (2, 3, "高収入なのに不合格(見逃し)")]):
    a.bar(x-w/2, [記録["そのまま"][i], 記録["揃えたあと"][i]], w, label="男性", color="#3A46A8")
    a.bar(x+w/2, [記録["そのまま"][j], 記録["揃えたあと"][j]], w, label="女性", color="#E08A2B")
    a.set_xticks(x); a.set_xticklabels(["そのまま", "揃えたあと"])
    a.set_ylabel("%"); a.set_title(題); a.legend(); a.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

print("偏りは消えたのではなく、場所を変えただけではありませんか。")
print()
print("合格率を揃えると、誤りの率が揃わなくなる。")
print("誤りの率を揃えると、合格率が揃わなくなる。")
print()
print("これは調整不足ではありません。もとの割合が違う限り、")
print("両方を同時に満たすことはできないと数学的に示されています(不可能性定理)。")

> **今日いちばん持ち帰ってほしいこと**
>
> 「公平にしてください」は、**そのままでは実行できない指示**です。
>
> 合格率を揃えるのか、誤りの率を揃えるのか、見逃しを減らすのか。
> どれも「公平」と呼べますが、**もとの割合が違う限り、同時には満たせません**(不可能性定理)。
>
> だから技術者は「公平にしました」と言えません。
> 言えるのは「**この定義で、この数字にしました**」までです。
> どの定義を選ぶかは、社会が決めることです。

## 5. 自分で試す

**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題用)=====
# (a) 深さを変えて、どこまでなら「読める」かを自分で判断する
print(f"{'深さ':>6}{'葉の数':>10}{'正解率':>10}")
print("-" * 28)
for d in range(2, 11):
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    print(f"{d:>6}{m.get_n_leaves():>10,}{m.score(Xte,yte)*100:>9.2f}%")
print(f"{'森':>6}{sum(t.get_n_leaves() for t in 森.estimators_):>10,}{森.score(Xte,yte)*100:>9.2f}%")
print()
print("あなたなら、どこまでを「読める」と考えますか。理由も書いてください。")

# (b) 別の属性(人種)でも、同じ検査をしてみる
人種 = (df["race"] == "White").astype(int)
_, _, _, _, 人tr, 人te = train_test_split(X, y, 人種, test_size=0.3, random_state=0, stratify=y)
print("\n" + "=" * 46)
for 名, 印 in [("White", 1), ("それ以外", 0)]:
    g = 人te == 印
    print(f"{名:<10} 高収入と判定 {予測[g].mean()*100:5.1f}%   実際 {yte[g].mean()*100:5.1f}%   (n={g.sum():,})")
m2 = RandomForestClassifier(n_estimators=50, random_state=0, n_jobs=-1).fit(Xtr, 人tr)
print(f"\n9項目から人種を予測: 正解率 {m2.score(Xte,人te)*100:.1f}%"
      f"(でたらめなら {max(人te.mean(),1-人te.mean())*100:.1f}%)")

## 6. 提出課題(manaba)

**締切:次回講義の前日 23:59**(次回は最終回・総まとめです)

### 課題1:公平さを検討する
ある企業が、AI で採用の書類選考を行いました(数字は架空)。

```
・入力は学歴、職務経歴、資格。性別は入力していない
・通過率:男性 22%、女性 9%
・企業の説明:「性別は使っていないので、公平です」
```

- (a) この説明は妥当ですか。**今日の実測を根拠に**答えなさい
- (b) 「性別の情報を使っていない」ことを確かめるには、何を測ればよいですか
- (c) 通過率を両者で揃えれば、問題は解決しますか

### 課題2:ノートブックを動かして報告する
- (a) 決定木の深さ別の正解率と葉の数の表(森との比較つき)
- (b) 3つの方法の重要度の順位表と、食い違いがいちばん大きい項目
- (c) 属性別の合格率・誤判定率・見逃し率を、**揃える前と後の両方**で

> (c) で片方だけの答案は評価を下げます。
> 両方ないと「偏りが場所を変えた」ことが見えないからです。

### 課題3:次の主張を批判する(300字以上)

> 「AI の判断は、重要度を表示すれば説明できます。また、保護属性を入力から除けば差別は起きません。
> これで説明責任も公平性も満たせるので、あとは精度を上げるだけです。」

考える材料:
- 「重要度」は一意に決まりますか。今日の実測ではどうでしたか
- 「保護属性を除けば差別は起きない」を、今日のどの数字が否定しますか
- 「公平性を満たす」は、そもそも一意に定義できますか
- 「精度を上げるだけ」でよいですか。今日、精度と説明可能性は取引関係にありましたか

**加点**:「では何を記録に残すべきか」まで書けると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。